<a href="https://colab.research.google.com/github/Exton-wz/PCVK-Ganjil-2026/blob/main/Week_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Week 6 - Filter Spasial


In [2]:
from google.colab import drive

drive.mount('/content/drive')

import numpy as np
import matplotlib.pyplot as plt
import cv2 as cv
import math
from google.colab.patches import cv2_imshow
from PIL import Image as im

Mounted at /content/drive


In [ ]:
# Fungsi tampil berdampingan
def show_side_by_side(images, titles, figsize=(15,5)):
    plt.figure(figsize=figsize)
    for i, (img, title) in enumerate(zip(images, titles)):
        if len(img.shape) == 2:  # grayscale
            plt.subplot(1, len(images), i+1)
            plt.imshow(img, cmap='gray')
        else:  # color
            plt.subplot(1, len(images), i+1)
            plt.imshow(cv.cvtColor(img, cv.COLOR_BGR2RGB))
        plt.title(title)
        plt.axis("off")
    plt.show()

### Membuat fungsi konvolusi

Parameter fungsi mengikuti yang ditampilkan pada modul: **image, kernel, stride, padding**. Implementasi berikut mengikuti pseudocode modul dan tidak menggunakan `cv.filter2D` untuk konvolusi manual.

In [ ]:
def convolution2d(image, kernel, stride, padding):
    # Padding citra
    image = image.astype(np.float32)
    kernel = kernel.astype(np.float32)

    padded = np.pad(image, ((padding, padding), (padding, padding)), mode='constant')

    # Ukuran hasil dibuat sama dengan ukuran citra input
    result = np.zeros_like(image, dtype=np.float32)

    for y in range(0, image.shape[0], stride):
        for x in range(0, image.shape[1], stride):
            value = 0
            for k1 in range(kernel.shape[0]):
                for k2 in range(kernel.shape[1]):
                    value += kernel[k1, k2] * padded[y + k1, x + k2]
            result[y, x] = value

    return result

### Verifikasi contoh perhitungan pada modul

Modul memberikan contoh perhitungan `g(2,2)` dengan bobot `1/4` dan hasil `7,75`.

In [ ]:
patch = np.array([[5,5,6],
                   [5,4,4],
                   [0,0,2]])

kernel = np.ones((3,3)) / 4

print(np.sum(patch * kernel))

7.75


### Verifikasi contoh matriks 5×5 dan kernel 3×3 pada modul

In [ ]:
f = np.array([[4,4,3,5,4],
               [6,6,5,5,2],
               [5,6,6,6,2],
               [6,7,5,5,3],
               [3,5,2,4,4]])

g = np.array([[0,-1,0],
               [-1,4,-1],
               [0,-1,0]])

# Bagian dalam citra (border diabaikan), sesuai tahapan pada modul
hasil = np.zeros((3,3))
for y in range(3):
    for x in range(3):
        hasil[y,x] = np.sum(f[y:y+3, x:x+3] * g)

print(hasil)

[[3. 0. 2.]
 [0. 2. 6.]
 [6. 0. 2.]]


### Load Image yang akan diproses

In [ ]:
img = cv.imread('/content/drive/MyDrive/PCVK 2026/mandrill.tiff')
img_gray = cv.cvtColor(img,cv.COLOR_BGR2GRAY)

### Menentukan kernel yang akan digunakan

In [ ]:
#image sharpen
kernel_sharpen = np.array([[0,-1,0],
                          [-1,5,-1],
                          [0,-1,0]])

### Memanggil fungsi konvolusi dan menampilkan hasil

In [ ]:
convolution2d(img_gray, kernel_sharpen, 1, 2)

array([[  0., -28., -22., ..., -55., -47., -13.],
       [-28.,  89.,  45., ..., 126., 142.,  -9.],
       [-29.,  66.,  21., ...,  76.,   1., -13.],
       ...,
       [-17.,  24.,  11., ...,  14.,  27.,  30.],
       [-27.,  93.,  33., ...,  31.,  37.,  28.],
       [ -7.,   3., -14., ...,  38.,  45.,  39.]], dtype=float32)

## 2. Image Filter: Average, Low Pass, High Pass, Sharpen, Emboss, Edge Detection, Gaussian Blur

Kernel yang digunakan mengikuti tabel pada Modul 6.

In [ ]:
# Average Filter 3x3
kernel_average = (1/9) * np.ones((3,3))

# Low Pass Filter
kernel_low_pass = (1/12) * np.array([[1,1,1],
                                     [1,4,1],
                                     [1,1,1]])

# High Pass Filter
kernel_high_pass = np.array([[-1,0,1],
                              [-1,0,3],
                              [-3,0,1]])

# Sharpen
kernel_sharpen = np.array([[0,-1,0],
                            [-1,5,-1],
                            [0,-1,0]])

# Emboss
kernel_emboss = np.array([[-2,-1,0],
                           [-1,1,1],
                           [0,1,2]])

# Left Sobel Edge Detection
kernel_sobel = np.array([[1,0,-1],
                         [2,0,-2],
                         [1,0,-1]])

# Canny Edge Detection (kernel yang ditampilkan pada tabel modul)
kernel_canny = np.array([[-1,-1,-1],
                         [-1,8,-1],
                         [-1,-1,-1]])

In [ ]:
average = convolution2d(img_gray, kernel_average, 1, 1)
low_pass = convolution2d(img_gray, kernel_low_pass, 1, 1)
high_pass = convolution2d(img_gray, kernel_high_pass, 1, 1)
sharpen = convolution2d(img_gray, kernel_sharpen, 1, 1)
emboss = convolution2d(img_gray, kernel_emboss, 1, 1)
sobel = convolution2d(img_gray, kernel_sobel, 1, 1)
canny_kernel = convolution2d(img_gray, kernel_canny, 1, 1)

show_side_by_side([average, low_pass, high_pass],
                  ['Average Filter', 'Low Pass Filter', 'High Pass Filter'])

show_side_by_side([sharpen, emboss, sobel, canny_kernel],
                  ['Sharpen', 'Emboss', 'Left Sobel Edge Detection', 'Canny Edge Detection'])

### 21x21 Gaussian Blur

Kode pembentukan kernel Gaussian mengikuti kode yang diberikan pada modul.

In [ ]:
kernel_size = 21
sigma=math.sqrt(kernel_size)
gaussian_kernel = cv.getGaussianKernel(kernel_size, sigma)
gauss_kernel = gaussian_kernel @ gaussian_kernel.transpose()

gaussian_blur = convolution2d(img_gray, gauss_kernel, 1, 10)
show_side_by_side([img_gray, gaussian_blur], ['Asli', '21x21 Gaussian Blur'])

# E. FILTER LIBRARY DAN FILTER MODERN

## Percobaan 1
Membuat Filter Gaussian, Sharpen, dan Canny menggunakan library `filter2d` dari OpenCV dan menerapkannya pada Image RGB.

In [ ]:
# Fungsi tampil berdampingan
def show_side_by_side(images, titles, figsize=(15,5)):
    plt.figure(figsize=figsize)
    for i, (img, title) in enumerate(zip(images, titles)):
        if len(img.shape) == 2:  # grayscale
            plt.subplot(1, len(images), i+1)
            plt.imshow(img, cmap='gray')
        else:  # color
            plt.subplot(1, len(images), i+1)
            plt.imshow(cv.cvtColor(img, cv.COLOR_BGR2RGB))
        plt.title(title)
        plt.axis("off")
    plt.show()

img = cv.imread('/content/drive/MyDrive/PCVK 2026/lena.jpg')
img_gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)

blur = cv.GaussianBlur(img, (7,7), 1)
edges = cv.Canny(cv.cvtColor(img, cv.COLOR_BGR2GRAY), 100, 200)
sharpen_kernel = np.array([[0,-1,0],[-1,5,-1],[0,-1,0]])
sharpened = cv.filter2D(img, -1, sharpen_kernel)
show_side_by_side([img, blur, sharpened, edges],
                  ['Asli', 'Gaussian Blur', 'Sharpened', 'Canny Edge Detection'])

## Percobaan 2
Filter modern dari library OpenCV: Bilateral Filtering dan Edge Preserving Filter.

In [ ]:
#Filter Modern dari OpenCV
# Bilateral Filter (edge-preserving)
bilateral = cv.bilateralFilter(img, 50, 100, 100)

# Edge Preserving Filter (alternatif Guided Filter)
edge_preserve = cv.edgePreservingFilter(img, flags=1, sigma_s=100, sigma_r=0.9)

show_side_by_side([img, bilateral, edge_preserve],
                  ['Asli', 'Bilateral Filter', 'Edge Preserving Filter'])

## Percobaan 3
Filtering pada CNN (bagian Feature Map). Jalankan kode beberapa kali dan perhatikan hasil outputnya.

In [ ]:
#Filter Feature Map yang digunakan pada CNN, lakukan running code bagian ini beberapa kali dan perhatikan hasilnya
import torch
import torch.nn as nn

class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 4, kernel_size=3, stride=1, padding=1)

    def forward(self, x):
        return self.conv1(x)

model = SimpleCNN()

# Ubah gambar ke tensor
img_tensor = torch.tensor(img_gray, dtype=torch.float32).unsqueeze(0).unsqueeze(0) / 255.0

# Hasil CNN
with torch.no_grad():
    features = model(img_tensor)

# Visualisasi feature maps
feature_maps = [features[0,i].numpy() for i in range(features.shape[1])]
show_side_by_side([img_gray] + feature_maps, ['Asli (Gray)'] + [f'Feature {i+1}' for i in range(len(feature_maps))])

## Percobaan 4
Efek Beauty dan Vintage menggunakan kombinasi filter tradisional.

In [ ]:
# =====================
# 1. Beauty Filter
# =====================

# Step 1: Smoothing kulit dengan bilateral filter
smooth = cv.bilateralFilter(img, d=15, sigmaColor=75, sigmaSpace=75)

# Step 2: Unsharp masking (pertajam mata/bibir)
gaussian = cv.GaussianBlur(smooth, (0,0), 3)
sharpened = cv.addWeighted(smooth, 1.5, gaussian, -0.5, 0)

# Step 3: Brightness & contrast
alpha = 1.2  # contrast
beta = 15    # brightness
beauty = cv.convertScaleAbs(sharpened, alpha=alpha, beta=beta)

# =====================
# 2. Old/Vintage Filter
# =====================

# Step 1: Sepia tone
sepia_kernel = np.array([[0.272, 0.534, 0.131],
                         [0.349, 0.686, 0.168],
                         [0.393, 0.769, 0.189]])
sepia = cv.transform(img, sepia_kernel)
sepia = np.clip(sepia, 0, 255).astype(np.uint8)

# Step 2: Vignette
rows, cols = img.shape[:2]
kernel_x = cv.getGaussianKernel(cols, cols*0.6)
kernel_y = cv.getGaussianKernel(rows, rows*0.6)
kernel = kernel_y * kernel_x.T
mask = kernel / kernel.max()
vignette = np.copy(sepia)
for i in range(3):
    vignette[:,:,i] = vignette[:,:,i] * mask

# Step 3: Noise/Grain
noise = np.random.normal(0, 15, vignette.shape).astype(np.int16)
old_img = np.clip(vignette.astype(np.int16) + noise, 0, 255).astype(np.uint8)

show_side_by_side([img, beauty, old_img],
                  ['Asli', 'Beauty Filter', 'Old/Vintage Filter'])

## Percobaan 5
Filter anime/cartoon menggunakan kombinasi filter tradisional.

In [ ]:
#Filter Anime / Cartoon
# Step 1: Edge detection (pakai median blur dulu agar gambar menjadi lebih halus)
gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)
gray_blur = cv.medianBlur(gray, 7)
edges = cv.adaptiveThreshold(gray_blur, 255,
                             cv.ADAPTIVE_THRESH_MEAN_C,
                             cv.THRESH_BINARY, 9, 9)

# Step 2: Bilateral filter untuk smoothing warna
color = cv.bilateralFilter(img, d=9, sigmaColor=200, sigmaSpace=200)

# Step 3: Gabungkan (cartoonize)
cartoon = cv.bitwise_and(color, color, mask=edges)

# Tampilkan
show_side_by_side([img, cartoon], ['Asli', 'Cartoon Filter'])

## Percobaan 6
Contoh Filter Malam.

In [ ]:
#Night Filter
img = cv.imread('/content/drive/MyDrive/PCVK 2026/djawatan.jpg')
img_gray = cv.cvtColor(img,cv.COLOR_BGR2GRAY)

# Step 1: Gelapkan (contrast turun, brightness negatif)
night = cv.convertScaleAbs(img, alpha=0.6, beta=-40)

# Step 2: Tambah bias biru
blue_tint = np.full_like(night, (50, 0, 100))  # BGR
night = cv.addWeighted(night, 0.8, blue_tint, 0.2, 0)

# Step 3: Efek glow di area terang dengan filter2D (blur kernel)
kernel = np.ones((15,15), np.float32) / 225
glow = cv.filter2D(night, -1, kernel)

# Kombinasikan asli + glow
night_glow = cv.addWeighted(night, 0.7, glow, 0.3, 0)

show_side_by_side([img, night, night_glow],
                  ['Asli', 'Night Filter', 'Night + Glow (filter2D)'])

## Percobaan 7
Filter Pagi dan Pagi ditambahkan efek kabut.

In [ ]:
#Filter Suasana pagi dan Kabut
# ============================
# Step 1: Kurangi kontras & cerahkan
# ============================
alpha = 0.9  # contrast
beta = 20    # brightness
soft = cv.convertScaleAbs(img, alpha=alpha, beta=beta)

# ============================
# Step 2: Tambahkan warm tone (kemerahan / oranye)
# ============================
warm_tint = np.full_like(soft, (40, 70, 120))  # BGR
pagi = cv.addWeighted(soft, 0.8, warm_tint, 0.2, 0)

# ============================
# Step 3: Tambahkan haze (kabut tipis) dengan filter2D
# ============================
# Kernel blur Gaussian-like untuk menciptakan efek kabut
kernel = cv.getGaussianKernel(3, 3)
kernel = kernel @ kernel.T  # jadikan 2D kernel
kabut = cv.filter2D(pagi, -1, kernel)

# tambah layer putih untuk kabut lebih nyata
white_layer = np.full_like(pagi, 255)
kabut = cv.addWeighted(kabut, 0.7, white_layer, 0.3, 0)

show_side_by_side([img, pagi, kabut],
                  ['Asli', 'Pagi', 'Pagi + Kabut'])

# Tugas Praktikum

## 1. Tugas Analisis: Perbandingan Mean Filter (Blur Biasa) vs. Median Filter pada Noise “Salt and Pepper”

a. Tambahkan noise bintik hitam-putih (Salt and Pepper) ke sebuah citra keabuan (grayscale).
b. Terapkan Mean Filter (3 x 3) dan Median Filter (3 x 3).
c. Amati dan analisis secara visual mengapa Median Filter lebih bersih dalam menghapus noise tersebut dibanding Mean Filter.

In [ ]:
# Masukkan lokasi gambar di Google Drive
path = '/content/drive/MyDrive/PCVK 2026/Gwagon_protrait.jpg'

# Membaca gambar
img = cv.imread(path)

# Mengubah gambar menjadi grayscale
img_gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)

# Membuat noise Salt and Pepper
noise = np.random.choice(
    [0, 255],
    img_gray.shape,
    p=[0.95, 0.05]
).astype(np.uint8)

salt_pepper = img_gray.copy()
salt_pepper[noise == 255] = 255

pepper = np.random.choice(
    [False, True],
    img_gray.shape,
    p=[0.95, 0.05]
)

salt_pepper[pepper] = 0

# Mean Filter 3x3
mean_filter = cv.blur(salt_pepper, (3, 3))

# Median Filter 3x3
median_filter = cv.medianBlur(salt_pepper, 3)

# Menampilkan semua hasil
gambar = [img_gray, salt_pepper, mean_filter, median_filter]
judul = [
    'Asli',
    'Salt and Pepper',
    'Mean Filter 3x3',
    'Median Filter 3x3']

plt.figure(figsize=(10, 8))

for i in range(4):
    plt.subplot(2, 2, i + 1)
    plt.imshow(gambar[i], cmap='gray')
    plt.title(judul[i])
    plt.axis('off')

plt.tight_layout()
plt.show()

## 2. Tugas Evaluasi: Pemilihan Ukuran Kernel untuk Sharpening

a. Buat kernel penajaman ukuran 3 x 3 dan 5 x 5.
b. Terapkan pada citra yang sedikit buram (blurry).
c. Evaluasi berdasarkan kejelasan tepi vs. noise/noise distortion.

In [ ]:
# Tugas 2
blurry = cv.GaussianBlur(img_gray, (0, 0), 2)

sharpen_3x3 = np.array([[0,-1,0],
                        [-1,5,-1],
                        [0,-1,0]])

sharpen_5x5 = np.array([[0,0,-1,0,0],
                        [0,-1,-1,-1,0],
                        [-1,-1,13,-1,-1],
                        [0,-1,-1,-1,0],
                        [0,0,-1,0,0]])

hasil_3x3 = cv.filter2D(blurry, -1, sharpen_3x3)
hasil_5x5 = cv.filter2D(blurry, -1, sharpen_5x5)

# Menampilkan hasil
hasil = [blurry, hasil_3x3, hasil_5x5]
judul = ['Blurry', 'Sharpen 3x3', 'Sharpen 5x5']

plt.figure(figsize=(12, 4))

for i in range(3):
    plt.subplot(1, 3, i + 1)
    plt.imshow(hasil[i], cmap='gray')
    plt.title(judul[i])
    plt.axis('off')

plt.tight_layout()
plt.show()

## 3. Tugas Kreasi: Filter Kartun Sederhana

Buat fungsi kustom `kartun(image)` yang menggabungkan Bilateral Filter, Adaptive Thresholding/Canny Edge, dan penggabungan keduanya.

In [ ]:
# Tugas 3
def kartun(image):
    gray = cv.cvtColor(image, cv.COLOR_BGR2GRAY)
    gray_blur = cv.medianBlur(gray, 7)

    edges = cv.adaptiveThreshold(
        gray_blur, 255,
        cv.ADAPTIVE_THRESH_MEAN_C,
        cv.THRESH_BINARY, 9, 9
    )

    color = cv.bilateralFilter(
        image, 9, 200, 200
    )

    cartoon = cv.bitwise_and(color, color, mask=edges)
    return cartoon

hasil_kartun = kartun(img)

# Menampilkan hasil
plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.imshow(cv.cvtColor(img, cv.COLOR_BGR2RGB))
plt.title('Asli')
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(cv.cvtColor(hasil_kartun, cv.COLOR_BGR2RGB))
plt.title('Cartoon Filter')
plt.axis('off')

plt.tight_layout()
plt.show()